# Cohort Longterm protocol → PyBaMM Experiment mapping

**Problem being addressed:** the previous 300-sim sweep used one fixed experiment protocol — full DoD at variable C-rate. Real cells span partial DoDs (`15_100`, `0_80`, `2_90`, etc.) as well. A Phase 2 DE fit against a mismatched simulation protocol will bias the identified θ (a cell cycled between 15%–100% SoC accumulates different SEI stress than a cell cycled 0%–100% at the same C-rate).

**What this notebook does:**

1. Pulls the actual (crate, drate, dod, nominal_capacity) from Athena's `cycle` table for each of the 13 cohort cells
2. Enumerates the unique protocols
3. Constructs the PyBaMM `Experiment` string for each unique protocol
4. Saves the mapping to `configs/cohort_experiment_protocols.yaml` so Phase 1/2 can consume it without re-querying Athena

**DoD notation:** `X_Y` = cell cycles between `X%` and `Y%` SoC. Examples:
- `0_100` = full DoD, 0→100→0 SoC each cycle
- `15_100` = 85% DoD, cycling between 15%–100% SoC (upper bound reached)
- `0_80` = 80% DoD, cycling between 0%–80% SoC (never fully charged)
- `2_90` = 88% DoD, both bounds moved in from full range

In [1]:
import os, yaml, warnings
from pathlib import Path

import boto3, awswrangler as wr
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings('ignore')
HERE = Path('/home/hj/Desktop/PINNs/Voltaris/Data_Exploration')
CONFIGS = Path('/home/hj/Desktop/PINNs/configs'); CONFIGS.mkdir(exist_ok=True)

os.environ.setdefault('AWS_CONFIG_FILE',             str(Path.home()/'Desktop/AWS/config'))
os.environ.setdefault('AWS_SHARED_CREDENTIALS_FILE', str(Path.home()/'Desktop/AWS/credentials'))
session = boto3.Session(profile_name='battery-turno', region_name='ap-south-1')

manifest = yaml.safe_load((HERE / 'training_cohort.yaml').read_text())
COHORT = {k: [str(c).zfill(4) for c in v] for k, v in manifest['cohort'].items()}
print(f'Cohort: {sum(len(v) for v in COHORT.values())} cells across {len(COHORT)} makes')

Cohort: 13 cells across 3 makes


In [2]:
where = ' OR '.join([
    f"(make='{m}' AND cell='{c}')"
    for m, cells in COHORT.items() for c in cells
])
sql = f"""SELECT make, cell, batch, crate, drate, dod, max_cap,
                 COUNT(*) AS n_cy
            FROM cycle
           WHERE test='Longterm' AND ({where})
           GROUP BY make, cell, batch, crate, drate, dod, max_cap
           ORDER BY make, cell, batch"""
proto = wr.athena.read_sql_query(sql=sql, database='rd_ts_cell_database',
                                     boto3_session=session, ctas_approach=False)
display(Markdown('### Per-cell protocol (both Athena batches)'))
display(proto)

### Per-cell protocol (both Athena batches)

,make,cell,batch,crate,drate,dod,max_cap,n_cy
0,CALB,0003,1,0.5C,0.5D,15_100,72,201
1,CALB,0003,2,0.5C,0.5D,15_100,72,201
2,CALB,0008,1,0.25C,0.25D,15_100,72,201
3,CALB,0008,2,0.25C,0.25D,15_100,72,201
4,CALB,0009,1,0.25C,0.25D,15_100,72,201
5,CALB,0009,2,0.25C,0.25D,15_100,72,201
6,CALB,0010,1,0.25C,0.25D,0_100,72,201
7,CALB,0010,2,0.25C,0.25D,0_100,72,200
8,CALB,0015,1,0.5C,0.5D,0_100,72,201
9,CALB,0015,2,0.5C,0.5D,0_100,72,201


In [3]:
# Assert each cell has consistent protocol across its batches (otherwise
# stitching them into one Experiment is unsafe)
def _proto_key(row):
    return (row['crate'], row['drate'], row['dod'])

conflict = (proto.groupby(['make','cell'])
                 .apply(lambda g: g[['crate','drate','dod']].drop_duplicates())
                 .groupby(level=[0,1]).size())
if (conflict > 1).any():
    display(Markdown('### ⚠ Protocol INCONSISTENCY across batches:'))
    display(conflict[conflict > 1])
else:
    display(Markdown('✓ Every cohort cell was cycled with the **same protocol in both Athena batches** — safe to represent as one Experiment per cell.'))

# Collapse to one row per cell (protocol shared across batches)
per_cell = (proto.sort_values(['make','cell','batch'])
                  .drop_duplicates(subset=['make','cell'], keep='first')
                  [['make','cell','crate','drate','dod','max_cap']]
                  .reset_index(drop=True))
display(Markdown('### One row per cell (protocol collapsed)'))
display(per_cell)

✓ Every cohort cell was cycled with the **same protocol in both Athena batches** — safe to represent as one Experiment per cell.

### One row per cell (protocol collapsed)

,make,cell,crate,drate,dod,max_cap
0,CALB,0003,0.5C,0.5D,15_100,72
1,CALB,0008,0.25C,0.25D,15_100,72
2,CALB,0009,0.25C,0.25D,15_100,72
3,CALB,0010,0.25C,0.25D,0_100,72
4,CALB,0015,0.5C,0.5D,0_100,72
5,EVE,0002,0.5C,0.5D,0_100,105
6,EVE,0004,0.5C,0.5D,2_100,105
7,EVE,0008,0.5C,0.5D,2_90,105
8,REPT,0004,0.25C,0.25D,0_80,150
9,REPT,0007,0.25C,0.25D,0_100,150


In [4]:
unique = per_cell[['make','crate','drate','dod','max_cap']].drop_duplicates() \
               .sort_values(['make','crate','drate','dod']).reset_index(drop=True)
unique['proto_id'] = unique.apply(
    lambda r: f"{r['make']}_{r['crate']}_{r['drate']}_{r['dod']}", axis=1)
unique['n_cohort_cells'] = unique.apply(
    lambda r: len(per_cell[(per_cell.make==r['make'])
                             & (per_cell.crate==r['crate'])
                             & (per_cell.drate==r['drate'])
                             & (per_cell.dod==r['dod'])]), axis=1)
unique['cells'] = unique.apply(
    lambda r: sorted(per_cell[(per_cell.make==r['make'])
                                & (per_cell.crate==r['crate'])
                                & (per_cell.drate==r['drate'])
                                & (per_cell.dod==r['dod'])]['cell'].tolist()), axis=1)
display(Markdown(f'### {len(unique)} unique protocols across the 13-cell cohort'))
display(unique[['proto_id','make','crate','drate','dod','max_cap','n_cohort_cells','cells']])

### 11 unique protocols across the 13-cell cohort

,proto_id,make,crate,drate,dod,max_cap,n_cohort_cells,cells
0,CALB_0.25C_0.25D_0_100,CALB,0.25C,0.25D,0_100,72,1,[0010]
1,CALB_0.25C_0.25D_15_100,CALB,0.25C,0.25D,15_100,72,2,"[0008, 0009]"
2,CALB_0.5C_0.5D_0_100,CALB,0.5C,0.5D,0_100,72,1,[0015]
3,CALB_0.5C_0.5D_15_100,CALB,0.5C,0.5D,15_100,72,1,[0003]
4,EVE_0.5C_0.5D_0_100,EVE,0.5C,0.5D,0_100,105,1,[0002]
5,EVE_0.5C_0.5D_2_100,EVE,0.5C,0.5D,2_100,105,1,[0004]
6,EVE_0.5C_0.5D_2_90,EVE,0.5C,0.5D,2_90,105,1,[0008]
7,REPT_0.25C_0.25D_0_100,REPT,0.25C,0.25D,0_100,150,1,[0007]
8,REPT_0.25C_0.25D_0_80,REPT,0.25C,0.25D,0_80,150,2,"[0004, 0012]"
9,REPT_0.5C_0.5D_0_100,REPT,0.5C,0.5D,0_100,150,1,[0057]


In [5]:
# LFP voltage bounds
V_CHARGE_CUTOFF   = 3.65    # V
V_DISCHARGE_CUTOFF = 2.50    # V
CV_TAIL_CURRENT_C = 50       # 'until C/50'
REST_MINUTES      = 10

def _c_rate_str(rate_field):
    """'0.5C' -> 0.5 (float); '0.25D' -> 0.25."""
    return float(str(rate_field).rstrip('CD'))

def _dod_window(dod):
    """'15_100' -> (0.15, 1.00)"""
    lo, hi = dod.split('_')
    return float(lo)/100.0, float(hi)/100.0

def _cycle_time_h(dod_frac, c_rate):
    """Hours to cycle the DoD span at the given C-rate."""
    return dod_frac / c_rate

def build_experiment(row):
    """Return a PyBaMM Experiment step list for one unique protocol."""
    c_c = _c_rate_str(row['crate'])
    c_d = _c_rate_str(row['drate'])
    lo, hi = _dod_window(row['dod'])
    dod_frac = hi - lo
    t_chg_h = _cycle_time_h(dod_frac, c_c)
    t_dis_h = _cycle_time_h(dod_frac, c_d)

    steps = []
    # ---- Charge ----
    if hi >= 0.999:
        # Full charge — use CC-CV protocol to reach top-of-charge
        steps.append(f"Charge at {c_c}C until {V_CHARGE_CUTOFF} V")
        steps.append(f"Hold at {V_CHARGE_CUTOFF} V until C/{CV_TAIL_CURRENT_C}")
    else:
        # Partial charge — CC only, time-limited (voltage cutoff as safety)
        steps.append(f"Charge at {c_c}C for {t_chg_h:.3f} hours "
                      f"or until {V_CHARGE_CUTOFF} V")
    steps.append(f"Rest for {REST_MINUTES} minutes")
    # ---- Discharge ----
    if lo <= 0.001:
        # Full discharge — voltage cutoff
        steps.append(f"Discharge at {c_d}C until {V_DISCHARGE_CUTOFF} V")
    else:
        # Partial discharge — time-limited (voltage cutoff as safety)
        steps.append(f"Discharge at {c_d}C for {t_dis_h:.3f} hours "
                      f"or until {V_DISCHARGE_CUTOFF} V")
    steps.append(f"Rest for {REST_MINUTES} minutes")
    return steps, dict(c_charge=c_c, c_discharge=c_d,
                        soc_low=lo, soc_high=hi, dod_frac=dod_frac,
                        cycle_time_charge_h=round(t_chg_h, 4),
                        cycle_time_discharge_h=round(t_dis_h, 4))

records = []
for _, r in unique.iterrows():
    steps, meta = build_experiment(r)
    records.append({
        'proto_id':          r['proto_id'],
        'make':              r['make'],
        'nominal_capacity_Ah': int(r['max_cap']),
        'n_cohort_cells':    int(r['n_cohort_cells']),
        'cells':             r['cells'],
        **meta,
        'V_charge_cutoff':   V_CHARGE_CUTOFF,
        'V_discharge_cutoff': V_DISCHARGE_CUTOFF,
        'rest_minutes':      REST_MINUTES,
        'cv_tail_current':   f'C/{CV_TAIL_CURRENT_C}',
        'experiment_steps':  steps,
        'source_crate':      r['crate'],
        'source_drate':      r['drate'],
        'source_dod':        r['dod'],
    })
protocol_df = pd.DataFrame(records)
display(Markdown(f'### Constructed PyBaMM Experiment step lists ({len(protocol_df)} protocols)'))
for _, r in protocol_df.iterrows():
    display(Markdown(f"**{r['proto_id']}** · {r['n_cohort_cells']} cell(s): `{r['cells']}` · SoC window `{r['soc_low']:.2f}–{r['soc_high']:.2f}` · DoD `{r['dod_frac']*100:.0f}%`"))
    for step in r['experiment_steps']:
        display(Markdown(f"  1. `{step}`"))


### Constructed PyBaMM Experiment step lists (11 protocols)

**CALB_0.25C_0.25D_0_100** · 1 cell(s): `['0010']` · SoC window `0.00–1.00` · DoD `100%`

  1. `Charge at 0.25C until 3.65 V`

  1. `Hold at 3.65 V until C/50`

  1. `Rest for 10 minutes`

...

**REPT_0.5C_0.5D_0_80** · 1 cell(s): `['0046']` · SoC window `0.00–0.80` · DoD `80%`

  1. `Charge at 0.5C for 1.600 hours or until 3.65 V`

  1. `Rest for 10 minutes`

  1. `Discharge at 0.5C until 2.5 V`

  1. `Rest for 10 minutes`

In [6]:
# Persist per-cell + per-protocol mapping
yaml_out = {
    'version': 1,
    'source': 'cohort_experiment_protocols.ipynb (Athena cycle table)',
    'defaults': {
        'V_charge_cutoff':    V_CHARGE_CUTOFF,
        'V_discharge_cutoff': V_DISCHARGE_CUTOFF,
        'rest_minutes':       REST_MINUTES,
        'cv_tail_current':    f'C/{CV_TAIL_CURRENT_C}',
    },
    'protocols': {
        r['proto_id']: {
            'make':              r['make'],
            'nominal_capacity_Ah': int(r['nominal_capacity_Ah']),
            'c_charge':          r['c_charge'],
            'c_discharge':       r['c_discharge'],
            'soc_low':           r['soc_low'],
            'soc_high':          r['soc_high'],
            'dod_frac':          r['dod_frac'],
            'cycle_time_charge_h':    r['cycle_time_charge_h'],
            'cycle_time_discharge_h': r['cycle_time_discharge_h'],
            'source_labels': {
                'crate': r['source_crate'], 'drate': r['source_drate'],
                'dod':   r['source_dod'],
            },
            'experiment_steps': r['experiment_steps'],
            'cohort_cells':     r['cells'],
        }
        for _, r in protocol_df.iterrows()
    },
    'cell_to_protocol': {
        f"{r['make']}_{r['cell']}": next(
            p['proto_id'] for _, p in protocol_df.iterrows()
            if r['cell'] in p['cells'] and p['make'] == r['make']
        )
        for _, r in per_cell.iterrows()
    },
}

target = CONFIGS / 'cohort_experiment_protocols.yaml'
target.write_text(yaml.safe_dump(yaml_out, sort_keys=False, allow_unicode=True))
print(f'Wrote: {target}')
print(f'Contains {len(yaml_out["protocols"])} unique protocols mapped to {len(yaml_out["cell_to_protocol"])} cohort cells')

Wrote: /home/hj/Desktop/PINNs/configs/cohort_experiment_protocols.yaml
Contains 11 unique protocols mapped to 13 cohort cells


In [7]:
# Show a concrete Phase 1/2 usage snippet
hint = '''
# Usage from Phase 2 (per-cell DE fit)
import yaml, pybamm
cfg = yaml.safe_load(open("configs/cohort_experiment_protocols.yaml"))

def cell_experiment(make: str, cell: str, n_cycles: int) -> pybamm.Experiment:
    proto_id = cfg["cell_to_protocol"][f"{make}_{cell}"]
    steps    = cfg["protocols"][proto_id]["experiment_steps"]
    return pybamm.Experiment([tuple(steps)] * n_cycles)

# example
exp = cell_experiment("CALB", "0008", n_cycles=150)  # 15_100 @ 0.25C
'''
print(hint)


# Usage from Phase 2 (per-cell DE fit)
import yaml, pybamm
cfg = yaml.safe_load(open("configs/cohort_experiment_protocols.yaml"))

def cell_experiment(make: str, cell: str, n_cycles: int) -> pybamm.Experiment:
    proto_id = cfg["cell_to_protocol"][f"{make}_{cell}"]
    steps    = cfg["protocols"][proto_id]["experiment_steps"]
    return pybamm.Experiment([tuple(steps)] * n_cycles)

# example
exp = cell_experiment("CALB", "0008", n_cycles=150)  # 15_100 @ 0.25C

